In [7]:
import os
from kaggle_secrets import UserSecretsClient
usc = UserSecretsClient()
for k in ["REDDIT_CLIENT_ID","REDDIT_CLIENT_SECRET","REDDIT_USER_AGENT","ANON_SALT"]:
    val = usc.get_secret(k)
    if val: os.environ[k] = val; print(f"{k}: loaded")
    else:   print(f"{k}: MISSING — add it in Add-ons → Secrets")

REDDIT_CLIENT_ID: loaded
REDDIT_CLIENT_SECRET: loaded
REDDIT_USER_AGENT: loaded
ANON_SALT: loaded


In [8]:
!pip install praw prawcore langdetect -q
!pip -q install praw fastparquet
print("✓ Packages installed")

✓ Packages installed


In [9]:
import os, re, json, time, random, hashlib
from datetime import datetime, timedelta, timezone
from typing import List, Dict, Any, Iterable, Optional, Tuple

import praw
import prawcore

In [10]:
try:
    from langdetect import detect
except Exception:
    detect = None

In [ ]:
# ====== CONFIG ===========

SAMPLE_FILE_PREFIX     = "reddit_pairs_3rd"
OUT_JSONL              = f"{SAMPLE_FILE_PREFIX}.jsonl"

# --- Generational subreddits
SUBS_BOOM = [
    "BoomerTears", "okboomer", "terriblefacebookmemes",
    "BoomersBeingFools", "boomerhumor", "BoomersBeingCools",
    "BoomerCringe", "goodboomerhumor", "BoomersAreTumors", "BoomerLogic"
]
SUBS_GENZ = [
    "GenZ", "genzmoment", "OlderGenZ", "GenZHumor",
    "MiddleGenZ", "GenZLiberals", "Younger_GenZ", "GenZMemes",
    "GenZIndia", "EarlyGenZ"
]
SUBS_MILL = [
    "Millennials", "millenials", "Older_Millennials", "DeathByMillennial",
    "Millennialshumor", "BlameMillennials", "DamnMillenials", "SecondWaveMillennials"
]
SUBS_GENX = [
    "GenX", "GenXVibes", "GenXWomen", "GenX_Culture", "GenerationJones",
    "GenXTalk", "AskGenX", "Xennials", "Younger_GenX", "YoungerGenX"
]
SUBS_CORE = ["generations", "generationology", "AskReddit", "TrueOffMyChest", "relationships"]

GROUPS_TO_SUBS = {
    "boomers": SUBS_BOOM,
    "genx": SUBS_GENX,
    "millennials": SUBS_MILL,
    "genz": SUBS_GENZ,
}

# --- Targets & limits
TARGET_PER_GROUP       = 1000
TIME_WINDOW_YEARS      = 3
LANG_EN_ONLY           = True
INCLUDE_DEEPER_REPLIES = True      # also collect child->grandchild
USE_CORE_FALLBACK      = True      # use core subs if needed
MAX_SUBMISSIONS_PER_SUB= 120       # per subreddit per pass (mix of new/hot)
TOPLEVEL_CAP_PER_POST  = 120       # cap top-level comments scanned per post
LIMIT_PER_AUTHOR_PER_SUB = 10
CHECKPOINT_EVERY       = 200

GROUP_PATTERNS = {
    "boomers": [
        r"\bboomer(s)?\b", r"\bbaby\s*boomer(s)?\b", r"\bok\s*boomer\b",
    ],
    "genx": [
        r"\bgen\s*-?\s*x\b", r"\bgeneration\s*x\b", r"\bgenx\b",
        r"\bx-?ers?\b", r"\bxennial(s)?\b"
    ],
    "millennials": [
        r"\bmillennial(s)?\b", r"\bmillenial(s)?\b", r"\bgen\s*-?\s*y\b",
    ],
    "genz": [
        r"\bgen\s*-?\s*z\b", r"\bgen\s*z['’]s\b", r"\bzoomers?\b", r"\bzoomer['’]s\b",
    ],
}

# --- False-positive guards (avoid off-domain hits)
NEGATIVE_PATTERNS = [
    r"\bboomer\s+shooter\b", r"\bboomerang(s)?\b",
    r"\bzoom\b", r"\bzoom(ed|ing)?\b", r"\bzoom[- ]?call\b",
    r"\bmillennium\b", r"\bmillenary\b", r"\bbooming\b",
    r"\bx[- ]?men\b", r"\bxbox\b", r"\bmac\s?os\s?x\b",
    r"\bgenx\s+chemical\b", r"\bchemours\b", r"\bpfas\b",
]

# --- Packaging & counter-moves heuristics — light regex proxies
#     Packaging: genericity, quantifiers, negation, noun vs adj label, hedges/boosters, abstraction-ish
QUANTIFIERS = r"\b(all|always|never|none|every|most|many|some|usually|often|typically|tend to)\b"
NEGATION    = r"\b(not|no|never|isn['’]t|aren['’]t|don['’]t|doesn['’]t|can['’]t|won['’]t)\b"
HEDGES      = r"\b(maybe|perhaps|probably|likely|seems|I think|I guess|kinda|sort of|somewhat)\b"
BOOSTERS    = r"\b(clearly|obviously|definitely|literally|undeniably|for sure)\b"
#     Counter-moves in replies
ADV_CONNECT = r"\b(but|however|although|though|yet)\b"
BROADENING  = r"\b(lots of people|others? (too|also)|everyone does|people in general|all ages)\b"
NOT_ALL     = r"\b(not all|not everyone|some of them|many of them)\b"
EVIDENCE    = r"\b(source\??|citation\??|link\??|prove it|stats?\b|according to)\b"
COUNTEREX   = r"\b(I know (a|an|several)|my (mom|dad|friend|coworker)|for example|e\.g\.)\b"
SARC        = r"(?:/s\b|yeah right|sure,\s*jan|as if|right\.)"

# ====== UTILITIES =========

def validate_credentials():
    for k in ["REDDIT_CLIENT_ID","REDDIT_CLIENT_SECRET","REDDIT_USER_AGENT"]:
        if not os.getenv(k):
            raise ValueError(f"Missing {k}. Set environment variables.")
    print("Credentials: OK")

def make_reddit():
    return praw.Reddit(
        client_id=os.getenv("REDDIT_CLIENT_ID"),
        client_secret=os.getenv("REDDIT_CLIENT_SECRET"),
        user_agent=os.getenv("REDDIT_USER_AGENT","IM-gen-extractor/0.3"),
        ratelimit_seconds=60,
    )

def compile_many(patts: List[str]) -> List[re.Pattern]:
    return [re.compile(p, re.I) for p in patts]

NEG_PATTERNS = compile_many(NEGATIVE_PATTERNS)
GROUP_COMPILED = {g: compile_many(ps) for g,ps in GROUP_PATTERNS.items()}

def mentions_any_group(text: str) -> Optional[str]:
    t = text or ""
    for g, pats in GROUP_COMPILED.items():
        if any(p.search(t) for p in pats):
            if any(n.search(t) for n in NEG_PATTERNS):
                return None
            return g
    return None

def mentions_group(text: str, gname: str) -> bool:
    t = text or ""
    return any(p.search(t) for p in GROUP_COMPILED[gname]) and not any(n.search(t) for n in NEG_PATTERNS)

def is_english(text: str) -> bool:
    if not LANG_EN_ONLY or not text:
        return True
    if detect is None:
        if len(text) < 20: return True
        ascii_ratio = sum(ch.isascii() for ch in text) / len(text)
        common = {"the","a","is","to","and","of","in","that","it","for"}
        words = set(re.findall(r"[a-zA-Z]+", text.lower()))
        return ascii_ratio > 0.95 and bool(words & common)
    try:
        return detect(text) == "en"
    except Exception:
        return False

def is_valid(body: Optional[str]) -> bool:
    if not body: return False
    b = body.strip()
    if len(b) < 10: return False
    lb = b.lower()
    return lb not in {"[deleted]","[removed]","[deleted by user]"}

def norm_for_dedup(s: str) -> str:
    s = s or ""
    s = s.lower()
    s = re.sub(r"\s+"," ",s).strip()
    s = re.sub(r"[^\w\s]","",s)
    return s

def pair_hash(parent_text: str, child_text: str) -> str:
    s = norm_for_dedup(parent_text) + "||" + norm_for_dedup(child_text)
    return hashlib.md5(s.encode("utf-8")).hexdigest()

def safe_call(thunk, label="API", retries=3):
    for i in range(retries):
        try:
            return thunk()
        except (prawcore.exceptions.RequestException,
                prawcore.exceptions.ResponseException,
                prawcore.exceptions.ServerError) as e:
            if i == retries-1: raise
            wait = (2**i)+random.uniform(0,1.0)
            print(f"{label} error, retry in {wait:.1f}s: {type(e).__name__}")
            time.sleep(wait)
    return None

# === METHODS ==
# Packaging & counter-moves

def detect_packaging(txt: str, gname: str) -> Dict[str, Any]:
    t = (txt or "").lower()
    group_pat = r"(?:boomer(?:s)?|baby\s*boomer(?:s)?|gen\s*-?\s*x|generation\s*x|genx|x-?ers?|xennial(?:s)?|millennial(?:s)?|millenial(?:s)?|gen\s*-?\s*y|gen\s*-?\s*z|zoomer(?:s)?)"
    copula = r"(?:are|is|be|tend to|usually|often)"
    generic = bool(re.search(rf"\b{group_pat}\b[^.?!,;]{{0,80}}\b{copula}\b", t) or
                   re.search(rf"\b(all|most|some|many|every|no)\b[^.?!,;]{{0,40}}\b{group_pat}\b", t))
    quant = bool(re.search(QUANTIFIERS, t))
    neg   = bool(re.search(NEGATION, t))
    hedge = bool(re.search(HEDGES, t))
    boost = bool(re.search(BOOSTERS, t))
    nounish = bool(re.search(r"\b(boomers|millennials|xers|xennials|zoomers)\b", t))
    adjish  = bool(re.search(r"\b(boomerish|millennial|gen[- ]?[xyz]-?ish)\b", t))
    traitish = bool(re.search(r"\b(lazy|entitled|smart|dumb|toxic|fragile|resilient|innovative|greedy|selfish)\b", t))
    eventive = bool(re.search(r"\b(work|study|learn|vote|buy|rent|use|code|drive|retire|own)\b", t))
    abstraction = "trait/generic" if traitish and not eventive else ("event/behavior" if eventive and not traitish else "mixed")
    return {
        "generic": generic,
        "quantifiers": quant,
        "negation": neg,
        "hedges": hedge,
        "boosters": boost,
        "label_form": "noun" if nounish and not adjish else ("adjective" if adjish and not nounish else "mixed"),
        "abstraction": abstraction,
    }

def detect_countermoves(reply: str) -> Dict[str, Any]:
    t = (reply or "").lower()
    return {
        "adversative": bool(re.search(ADV_CONNECT, t)),
        "broadening":  bool(re.search(BROADENING, t)),
        "not_all":     bool(re.search(NOT_ALL, t)),
        "evidence":    bool(re.search(EVIDENCE, t)),
        "counterex":   bool(re.search(COUNTEREX, t)),
        "sarcasm":     bool(re.search(SARC, t)),
    }

# ====== FETCHING =========

def submissions_recent(reddit, subreddit: str, limit: int=MAX_SUBMISSIONS_PER_SUB) -> List[Any]:
    seen, out = set(), []
    sr = reddit.subreddit(subreddit)
    def take(gen, cap):
        cnt=0
        for s in gen:
            if getattr(s, "id", None) in seen: continue
            seen.add(s.id)
            out.append(s); cnt+=1
            if cnt>=cap: break
    try: take(sr.new(limit=limit//2), max(10, limit//2))
    except Exception: pass
    try: take(sr.hot(limit=limit//2), max(10, limit//2))
    except Exception: pass
    return out

def fetch_pairs_from_submission(submission,
                                target_group: str,
                                dedup: set,
                                per_author_counts: Dict[Tuple[str,str], int],
                                after_utc: int, before_utc: int) -> List[Dict[str, Any]]:
    out=[]
    ts = int(getattr(submission, "created_utc", 0) or 0)
    if not (after_utc <= ts <= before_utc):
        return out

    title = submission.title or ""
    selftext = submission.selftext or ""
    post_group = mentions_any_group(title + "\n" + selftext)

    # Expand comments only if post hints at group OR we’re scanning anyway for target_group
    try:
        submission.comment_sort = "top"
        submission.comment_limit = TOPLEVEL_CAP_PER_POST
        safe_call(lambda: submission.comments.replace_more(limit=0), label="replace_more")
    except Exception:
        return out

    # Iterate top-levels
    toplevel = submission.comments
    if hasattr(toplevel, "__getitem__"):
        toplevel = submission.comments[:TOPLEVEL_CAP_PER_POST]

    for parent in toplevel:
        if not getattr(parent, "is_root", False): continue
        if not is_valid(getattr(parent, "body","")): continue
        if not is_english(parent.body): continue

        # Parent must mention the target group OR the post does
        if not (mentions_group(parent.body, target_group) or (post_group==target_group)):
            continue

        # Per-author soft cap (subreddit-scoped)
        author_name = (str(parent.author).lower() if parent.author else "[deleted]")
        sub_name = submission.subreddit.display_name.lower()
        ak = (author_name, sub_name)
        per_author_counts.setdefault(ak, 0)
        if per_author_counts[ak] >= LIMIT_PER_AUTHOR_PER_SUB:
            continue

        parent_pkg = detect_packaging(parent.body, target_group)

        # Direct replies
        replies = getattr(parent, "replies", [])
        for child in replies:
            if not is_valid(getattr(child,"body","")): continue
            if not is_english(child.body): continue

            h = pair_hash(parent.body, child.body)
            if h in dedup: continue

            cm = detect_countermoves(child.body)

            rec = {
                "target_group": target_group,
                "subreddit": submission.subreddit.display_name,
                "post_id": submission.id,
                "post_title": title,
                "post_selftext": selftext,
                "parent_comment_id": parent.id,
                "child_comment_id": child.id,
                "parent_text": parent.body,
                "child_text": child.body,
                "parent_packaging": parent_pkg,
                "reply_counters": cm,
            }
            out.append(rec)
            dedup.add(h)
            per_author_counts[ak] += 1

            # One extra hop if enabled
            if INCLUDE_DEEPER_REPLIES and getattr(child,"replies",None):
                for gc in child.replies:
                    if not is_valid(getattr(gc,"body","")): continue
                    if not is_english(gc.body): continue
                    h2 = pair_hash(child.body, gc.body)
                    if h2 in dedup: continue
                    cm2 = detect_countermoves(gc.body)
                    rec2 = {
                        "target_group": target_group,
                        "subreddit": submission.subreddit.display_name,
                        "post_id": submission.id,
                        "post_title": title,
                        "post_selftext": selftext,
                        "parent_comment_id": child.id,
                        "child_comment_id": gc.id,
                        "parent_text": child.body,
                        "child_text": gc.body,
                        "parent_packaging": detect_packaging(child.body, target_group),
                        "reply_counters": cm2,
                    }
                    out.append(rec2)
                    dedup.add(h2)
    return out

def write_jsonl(path: str, recs: List[Dict[str,Any]], save_context: bool=True) -> int:
    ok=0
    with open(path,"a",encoding="utf-8") as f:
        for r in recs:
            try:
                if not save_context:
                    r = {k:v for k,v in r.items() if k not in {"post_title","post_selftext","parent_packaging","reply_counters"}}
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
                ok+=1
            except Exception as e:
                print("Write warn:", e)
    return ok

def round_robin(items: List[str]):
    i=0; L=len(items)
    while L:
        yield items[i % L]; i+=1

def collect_for_group(reddit, group: str, primary_subs: List[str],
                      after_utc: int, before_utc: int,
                      target_pairs: int, use_core: bool=True) -> List[Dict[str,Any]]:

    print(f"\n[{group}] → target {target_pairs} | subs: {', '.join(primary_subs[:4])}{'...' if len(primary_subs)>4 else ''}")
    out=[]; dedup=set(); per_author={}
    start=time.time(); processed=0

    # Phase 1: group subs
    rr = round_robin(primary_subs)
    empty_rounds=0; max_empty=max(2*len(primary_subs),6)

    while len(out) < target_pairs:
        sub = next(rr)
        submissions = submissions_recent(reddit, sub, limit=MAX_SUBMISSIONS_PER_SUB)
        random.shuffle(submissions)
        got=0
        for s in submissions:
            pairs = fetch_pairs_from_submission(s, group, dedup, per_author, after_utc, before_utc)
            if pairs:
                out.extend(pairs); got += len(pairs); processed += 1
                if len(out) % 100 == 0:
                    elapsed = time.time()-start
                    rate = (len(out)/elapsed*60) if elapsed>0 else 0
                    print(f"  [{group}] {len(out)}/{target_pairs} pairs ({rate:.1f}/min)  posts:{processed}")
                if len(out) % CHECKPOINT_EVERY == 0:
                    wrote = write_jsonl(OUT_JSONL, out[-CHECKPOINT_EVERY:])
                    print(f"  [{group}] checkpoint wrote {wrote}")
            if len(out) >= target_pairs:
                break
        if got==0:
            empty_rounds += 1
            if empty_rounds >= max_empty:
                print(f"  [{group}] Phase 1 exhausted; switching to core")
                break
        else:
            empty_rounds = 0
        time.sleep(1.0)

    # Phase 2: core subs
    if use_core and len(out) < target_pairs:
        rr2 = round_robin(SUBS_CORE)
        empty_rounds=0; max_empty=max(2*len(SUBS_CORE),6)
        while len(out) < target_pairs:
            sub = next(rr2)
            submissions = submissions_recent(reddit, sub, limit=MAX_SUBMISSIONS_PER_SUB)
            random.shuffle(submissions)
            got=0
            for s in submissions:
                pairs = fetch_pairs_from_submission(s, group, dedup, per_author, after_utc, before_utc)
                if pairs:
                    out.extend(pairs); got += len(pairs); processed += 1
                    if len(out) % 100 == 0:
                        elapsed = time.time()-start
                        rate = (len(out)/elapsed*60) if elapsed>0 else 0
                        print(f"  [{group}] {len(out)}/{target_pairs} pairs ({rate:.1f}/min)  posts:{processed}")
                    if len(out) % CHECKPOINT_EVERY == 0:
                        wrote = write_jsonl(OUT_JSONL, out[-CHECKPOINT_EVERY:])
                        print(f"  [{group}] checkpoint wrote {wrote}")
                if len(out) >= target_pairs:
                    break
            if got==0:
                empty_rounds += 1
                if empty_rounds >= max_empty:
                    print(f"  [{group}] core exhausted; stop")
                    break
            else:
                empty_rounds = 0
            time.sleep(1.0)

    return out[:target_pairs]

def main():
    print("="*60); print("Reddit 4-Gen Extractor (IM methods) v0.3"); print("="*60)
    validate_credentials()
    reddit = make_reddit()

    now = datetime.now(timezone.utc)
    after = int((now - timedelta(days=365*TIME_WINDOW_YEARS)).timestamp())
    before= int(now.timestamp())

    totals=0
    for g in ["genz","millennials","genx","boomers"]:
        pairs = collect_for_group(
            reddit=reddit,
            group=g,
            primary_subs=GROUPS_TO_SUBS[g],
            after_utc=after, before_utc=before,
            target_pairs=TARGET_PER_GROUP,
            use_core=USE_CORE_FALLBACK
        )
        wrote = write_jsonl(OUT_JSONL, pairs, save_context=True)   # flip to False for IDs+texts-only
        print(f"  [{g}] wrote {wrote}/{len(pairs)} → {OUT_JSONL}")
        totals += len(pairs)

    print("="*60); print(f"✓ Done. Total pairs written: {totals} → {OUT_JSONL}"); print("="*60)

if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\nInterrupted.")
    except Exception as e:
        print("Fatal:", type(e).__name__, e)


Reddit 4-Gen Extractor (IM methods) v0.3
Credentials: OK

[genz] → target 1000 | subs: GenZ, genzmoment, OlderGenZ, GenZHumor...
replace_more error, retry in 1.8s: TooManyRequests
replace_more error, retry in 1.6s: TooManyRequests
replace_more error, retry in 1.6s: TooManyRequests
replace_more error, retry in 1.4s: TooManyRequests
replace_more error, retry in 1.8s: TooManyRequests
replace_more error, retry in 1.9s: TooManyRequests
replace_more error, retry in 1.5s: TooManyRequests
replace_more error, retry in 1.3s: TooManyRequests
replace_more error, retry in 1.3s: TooManyRequests
